# Using the Visualizer

The Visualizer draws a resource, such as a deck, with everything on it and its state, so you can see what is going on: when running a protocol on a robot, or when developing one.

The Visualizer follows a resource store. It reads the resource from the store once, then draws every change any client of the store makes to it: a plate placed by hand, a tip picked up by a device, a well filled.

## A deck in a resource store

Make a store in this process and create a STARlet deck in it.

In [ ]:
from pylabrobot.resources.store import ResourceStore
from pylabrobot.visualizer import Visualizer

store = ResourceStore.in_memory()
deck = await store.create("STARLetDeck", name="deck")

Next, create a {class}`~pylabrobot.visualizer.visualizer.Visualizer`. It takes a handle to the resource to draw, here the deck.

In [ ]:
vis = Visualizer(resource=deck)
await vis.setup()

![The empty simulator](./img/visualizer/empty.png)

## Build the deck layout: Assigning plates and tips

Resources created in, or assigned to, the deck appear in the visualization.

In [ ]:
tip_car = await store.create("hamilton_tip_carrier_L5", name="tip carrier", parent=deck, track=15)
tip_racks = [
  await store.create(
    "hamilton_96_tiprack_1000uL_filter", name=f"tips_0{i + 1}", parent=tip_car[i]
  )
  for i in range(5)
]
tip_rack1, tip_rack2, tip_rack3, tip_rack4, tip_rack5 = tip_racks
for rack in tip_racks:
  await rack.empty()

In [ ]:
plt_car = await store.create("hamilton_plate_carrier_L5_ac", name="plate carrier", parent=deck, track=8)
plate_1, plate_2, plate_3 = [
  await store.create("cor_96_wellplate_360uL_Fb", name=f"plate_0{i + 1}", parent=plt_car[i])
  for i in range(3)
]

![The simulator after the resources have been assigned](./img/visualizer/assignment.png)

### Configuring the state of the deck

As with every PyLabRobot script, you can say what is on the deck before a method starts. The store keeps track of it, and the visualizer draws it.

### Tips

Use `fill` to place tips at all spots of a tip rack.

In [ ]:
await tip_rack1.fill()

`set_tip_state` says, spot by spot, whether a spot holds a tip.

In [ ]:
tip_rack4 = await store.get("tips_04", root=deck)
await tip_rack4.set_tip_state([True] * 48 + [False] * 48)

In [ ]:
await tip_rack3.set_tip_state(([True] * 8 + [False] * 8) * 6)

In [ ]:
await tip_rack2.set_tip_state(([True] * 16 + [False] * 16) * 3)

### Liquids

`set_well_volumes` sets the volume of each well of a plate, in uL.

In [ ]:
await plate_1.set_well_volumes([500] * 96)

In [ ]:
await plate_2.set_well_volumes([100, 500] * (96 // 2))

In the visualizer, you can see that the opacity of the well is proportional to how full the well is relative to its maximum volume.

![Simulator after the tips have been placed and the volumes adjusted](./img/visualizer/resources.png)

## Liquid handling

A liquid handler working on the same store - picking up tips, aspirating, dispensing - changes the resources it works on in the store, and the visualizer draws each change as the store makes it. All changes of one command arrive together, so a 96-channel pick-up is drawn at once.

## Shutting down

When you're done, stop the visualizer by calling {func}`~pylabrobot.visualizer.visualizer.Visualizer.stop`.

In [ ]:
await vis.stop()